# 03 - Train, Validation och Test Split

## Syfte

Syftet med denna notebook är att dela det labeled datasetet i separata train-, validation- och testdataset.

Eftersom målet är att bygga en modell som ska kunna generalisera till framtida orders behöver datan delas innan mer detaljerad analys och modellering görs.

Jag väljer en tidsbaserad uppdelning där äldre orders används för träning och nyare orders används för validation och test. På så sätt efterliknar uppdelningen bättre en verklig situation där modellen tränas på historisk data och används på framtida data.

Efter uppdelningen kommer jag att kontrollera datumintervall, target-fördelning och att samma `order_id` inte förekommer i flera dataset.

In [37]:
from pathlib import Path
import pandas as pd

# Artifact från Notebook 02
artifacts_dir = Path("artifacts")

# Läs in det labelerade datasetet
labeled_ml_table = pd.read_csv(
    artifacts_dir / "labeled_ml_table.csv"
)

print("Shape:", labeled_ml_table.shape)
print("Unique order_id:", labeled_ml_table["order_id"].nunique())
print(
    "Duplicate order_id:",
    labeled_ml_table["order_id"].duplicated().sum()
)

Shape: (96476, 21)
Unique order_id: 96476
Duplicate order_id: 0


### Resultat och tolkning

Det labeled datasetet har lästs in korrekt och innehåller 96 476 orders
och 21 kolumner.

Varje rad representerar en unik order och target-variabeln `late` finns
med i datasetet.

Innan uppdelningen görs kontrolleras datumkolumnerna. `order_purchase_timestamp`
kommer att användas för att sortera orders kronologiskt.

## Förbered orderdatum för tidsbaserad uppdelning

För att kunna dela upp datasetet kronologiskt behöver
`order_purchase_timestamp` behandlas som ett datum.

Vi använder orderns köpdatum som tidsvariabel eftersom uppdelningen
ska efterlikna ett verkligt scenario där modellen tränas på äldre
orders och utvärderas på nyare orders.

In [38]:
# Konvertera orderns köpdatum till datetime
labeled_ml_table["order_purchase_timestamp"] = pd.to_datetime(
    labeled_ml_table["order_purchase_timestamp"],
    errors="coerce"
)

# Kontrollera att inga datum saknas efter konverteringen
print(
    "Missing order purchase timestamps:",
    labeled_ml_table["order_purchase_timestamp"].isna().sum()
)

# Visa tidsperioden i datasetet
print(
    "Earliest order:",
    labeled_ml_table["order_purchase_timestamp"].min()
)

print(
    "Latest order:",
    labeled_ml_table["order_purchase_timestamp"].max()
)

Missing order purchase timestamps: 0
Earliest order: 2016-09-15 12:16:38
Latest order: 2018-08-29 15:00:37


### Resultat och beslut om split-metod

Alla orders har ett giltigt `order_purchase_timestamp`, vilket innebär att
hela det labelerade datasetet kan användas för en tidsbaserad uppdelning.

Datasetet sträcker sig från 2016-09-15 till 2018-08-29.

Vi väljer en kronologisk split där äldre orders används för träning och
nyare orders används för validation och test.

Detta efterliknar ett mer realistiskt ML-scenario där modellen tränas på
historiska data och sedan används på framtida orders.

Ingen stratifiering används eftersom den kronologiska ordningen ska bevaras.

In [39]:
# Sortera orders kronologiskt efter köpdatum
labeled_ml_table = (
    labeled_ml_table
    .sort_values("order_purchase_timestamp")
    .reset_index(drop=True)
)

# Beräkna gränser för 70 % train, 15 % validation och 15 % test
n = len(labeled_ml_table)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

# Skapa de tre dataset-delarna
train_data = labeled_ml_table.iloc[:train_end].copy()

validation_data = labeled_ml_table.iloc[
    train_end:validation_end
].copy()

test_data = labeled_ml_table.iloc[
    validation_end:
].copy()

print("Total rows:", n)
print("Train shape:", train_data.shape)
print("Validation shape:", validation_data.shape)
print("Test shape:", test_data.shape)

print(
    "Total rows after split:",
    len(train_data) + len(validation_data) + len(test_data)
)

Total rows: 96476
Train shape: (67533, 21)
Validation shape: (14471, 21)
Test shape: (14472, 21)
Total rows after split: 96476


## Kontroll av tidsperioder efter split

Efter den kronologiska uppdelningen kontrollerar vi tidsperioden i varje
dataset.

Syftet är att verifiera att train innehåller de äldsta orders, validation
senare orders och test de senaste orders. Detta säkerställer att den
tidsbaserade uppdelningen har genomförts korrekt.

In [40]:
# Kontrollera tidsperioden i varje dataset
for name, data in [
    ("Train", train_data),
    ("Validation", validation_data),
    ("Test", test_data)
]:
    print(f"{name}:")
    print("  Start:", data["order_purchase_timestamp"].min())
    print("  End:  ", data["order_purchase_timestamp"].max())
    print()

Train:
  Start: 2016-09-15 12:16:38
  End:   2018-04-15 20:07:56

Validation:
  Start: 2018-04-15 20:10:23
  End:   2018-06-21 07:50:39

Test:
  Start: 2018-06-21 08:29:29
  End:   2018-08-29 15:00:37



### Resultat: tidsperioder

Den kronologiska uppdelningen är korrekt.

- **Train:** 2016-09-15 till 2018-04-15
- **Validation:** 2018-04-15 till 2018-06-21
- **Test:** 2018-06-21 till 2018-08-29

Train innehåller de äldsta orders, validation innehåller senare orders
och test innehåller de senaste orders.

Detta bevarar den kronologiska ordningen och gör att modellen senare
kan utvärderas på data som ligger efter träningsdata i tiden.

## Kontroll av target-fördelning

Efter uppdelningen kontrollerar vi fördelningen av target-variabeln `late`
i train, validation och test.

Syftet är att se hur class imbalance ser ut i de tre tidsperioderna.
Eftersom vi använder en tidsbaserad split förväntar vi oss inte exakt
samma klassfördelning i alla delar.

In [41]:
# Kontrollera target-fördelningen i varje dataset
for name, data in [
    ("Train", train_data),
    ("Validation", validation_data),
    ("Test", test_data)
]:
    counts = data["late"].value_counts().sort_index()
    percentages = (
        data["late"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
    )

    print(f"{name}:")
    print("Antal:")
    print(counts)

    print("Procent:")
    print(percentages.round(2))
    print()

Train:
Antal:
late
0    61436
1     6097
Name: count, dtype: int64
Procent:
late
0    90.97
1     9.03
Name: proportion, dtype: float64

Validation:
Antal:
late
0    13698
1      773
Name: count, dtype: int64
Procent:
late
0    94.66
1     5.34
Name: proportion, dtype: float64

Test:
Antal:
late
0    13515
1      957
Name: count, dtype: int64
Procent:
late
0    93.39
1     6.61
Name: proportion, dtype: float64



### Resultat: target-fördelning

Alla tre dataset-delar har en tydlig class imbalance där majoriteten av
orders har `late = 0`.

- **Train:** 9,03 % sena orders (`late = 1`)
- **Validation:** 5,34 % sena orders
- **Test:** 6,61 % sena orders

Andelen sena orders varierar mellan tidsperioderna. Detta är förväntat
vid en kronologisk split eftersom ingen stratifiering används och den
verkliga förändringen över tid bevaras.

Class imbalance behöver därför tas hänsyn till senare vid val av
utvärderingsmått och modellering.

## Kontroll av order-overlap

Innan dataset-delarna sparas kontrollerar vi att samma `order_id` inte
förekommer i mer än en del.

Train, validation och test ska vara helt separata för att undvika
data leakage mellan dataset-delarna.

In [42]:
# Kontrollera att inga order_id förekommer i flera dataset-delar
train_ids = set(train_data["order_id"])
validation_ids = set(validation_data["order_id"])
test_ids = set(test_data["order_id"])

train_validation_overlap = len(train_ids & validation_ids)
train_test_overlap = len(train_ids & test_ids)
validation_test_overlap = len(validation_ids & test_ids)

print("Train / Validation overlap:", train_validation_overlap)
print("Train / Test overlap:", train_test_overlap)
print("Validation / Test overlap:", validation_test_overlap)

Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0


### Resultat: order-overlap

Ingen `order_id` förekommer i mer än en dataset-del.

Train, validation och test är därmed separata på ordernivå, vilket minskar
risken för data leakage mellan dataset-delarna.

## Spara train, validation och test

De tre dataset-delarna sparas som separata artifacts.

Dessa filer används i efterföljande notebooks. Den detaljerade EDA:n
kommer endast att göras på träningsdata för att undvika information
från validation och test under analys och feature engineering.

In [43]:
# Spara dataset-delarna som artifacts
train_path = artifacts_dir / "train.csv"
validation_path = artifacts_dir / "validation.csv"
test_path = artifacts_dir / "test.csv"

train_data.to_csv(train_path, index=False)
validation_data.to_csv(validation_path, index=False)
test_data.to_csv(test_path, index=False)

print("Saved:", train_path, train_data.shape)
print("Saved:", validation_path, validation_data.shape)
print("Saved:", test_path, test_data.shape)

print("\nFiles exist:")
print("Train:", train_path.exists())
print("Validation:", validation_path.exists())
print("Test:", test_path.exists())

Saved: artifacts\train.csv (67533, 21)
Saved: artifacts\validation.csv (14471, 21)
Saved: artifacts\test.csv (14472, 21)

Files exist:
Train: True
Validation: True
Test: True


## SISTA RESULTAT 

Det labelerade datasetet delades upp kronologiskt i train, validation
och test utifrån `order_purchase_timestamp`.

En tidsbaserad split valdes för att efterlikna ett realistiskt scenario
där modellen tränas på historiska orders och utvärderas på nyare data.
Därför användes ingen stratifiering.

Fördelningen blev:

- **Train:** 67 533 orders (70 %)
- **Validation:** 14 471 orders (15 %)
- **Test:** 14 472 orders (15 %)

Den kronologiska ordningen verifierades genom att kontrollera
tidsperioderna för varje dataset-del. Ingen `order_id` förekommer i mer
än en del, vilket minskar risken för data leakage.

Target-fördelningen visar class imbalance i samtliga dataset-delar.
Andelen sena orders (`late = 1`) är 9,03 % i train, 5,34 % i validation
och 6,61 % i test. Skillnaden mellan perioderna bevaras eftersom
uppdelningen är tidsbaserad.

De tre dataset-delarna sparades som:

- `artifacts/train.csv`
- `artifacts/validation.csv`
- `artifacts/test.csv`

Fortsatt detaljerad EDA ska göras endast på träningsdata. Validation
används senare för modellval och tuning, medan test hålls separat fram
till den slutliga utvärderingen.